# signalign corpus factory — Colab runner

Runs the signalign dataset factory (Demucs → VAD → lyrics-informed forced
alignment → calibrated confidence gate) on CC-BY / CC-BY-SA vocal tracks
with lyrics, producing gated, schema-valid aligned segments.

Source audio is **pre-fetched locally** (`scripts/fetch_jamendo.py` — the
Jamendo API returns an empty catalog to datacenter IPs, so fetching from
Colab does not work) and staged on Hugging Face under
`signalign-corpus-staging/source/`. This notebook downloads it from there
and does the GPU-heavy part.

**Runtime → Change runtime type → T4 GPU** before running.

Secrets needed (Colab left sidebar 🔑):
- `HF_TOKEN` — read+write, for pulling staged source and uploading results

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/Alcadramin/signalign.git
%cd signalign
!pip install -q demucs faster-whisper silero-vad soundfile

In [ ]:
import torch, torchaudio
assert torch.cuda.is_available(), 'enable the T4 runtime'
assert hasattr(torchaudio.functional, 'forced_align'), 'torchaudio too old'
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
REPO = 'Alcadramin/signalign-corpus-staging'
!hf download {REPO} --repo-type dataset --include 'source/*' --local-dir data_staged
!mkdir -p data && ln -sfn $(pwd)/data_staged/source data/jamendo_corpus
!ls data/jamendo_corpus/audio | wc -l

In [ ]:
config = '''
[input]
audio_dir = "data/jamendo_corpus/audio"
lyrics_dir = "data/jamendo_corpus/lyrics"
licenses_csv = "data/jamendo_corpus/tracks.csv"
source = "jamendo"
license = "unknown"

[output]
dir = "out/corpus"
'''
open('corpus.toml', 'w').write(config)
!python pipeline/run.py --config corpus.toml

In [ ]:
import json
for pile in ['keep', 'hard']:
    try:
        records = [json.loads(l) for l in open(f'out/corpus/{pile}.jsonl')]
        words = sum(len(r['words']) for r in records)
        hours = sum(r['duration'] for r in records) / 3600
        print(f'{pile}: {len(records)} segments, {words} words, {hours:.1f}h audio')
    except FileNotFoundError:
        print(pile, '- none')

## Upload results to Hugging Face

Uploads manifests + clips to a **private staging** dataset repo. The
public `signalign-corpus` release happens after human spot-check.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
REPO = 'Alcadramin/signalign-corpus-staging'
!hf repo create {REPO} --repo-type dataset --private -y 2>/dev/null || true
!cp data/jamendo_corpus/tracks.csv out/corpus/
!hf upload {REPO} out/corpus . --repo-type dataset --commit-message 'colab factory run'